# Notebook 20 — Capa Semántica y Reglas de Negocio
**Reto 1: Operational Intelligence**

**Propósito:** Traducir los hallazgos del EDA en artefactos gobernados que definan el contrato formal entre datos crudos, insight engine (NB 30) y chatbot (NB 40).  
**Input:** artefactos de `00_reto1_data_prep`, hallazgos de `10_reto1_eda`, archivos `config/*.yaml`.  
**Output:** validación del contrato semántico, catálogo enriquecido con estadísticas del dataset, reporte `reports/reto1/semantic_layer_report.{json,md}`.

> **Estado:** iteración inicial. Todas las direcciones y thresholds son *hipótesis de trabajo*. Nada está validado con negocio.

## 0. Setup

In [9]:
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings('ignore')

# ── root detection ─────────────────────────────────────────────────────────
def find_project_root(start: Path = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'data' / 'processed').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Project root not found')

ROOT     = find_project_root()
PROCESSED = ROOT / 'data' / 'processed'
INTERIM   = ROOT / 'data' / 'interim'
CONFIG    = ROOT / 'config'
REPORTS   = ROOT / 'reports' / 'reto1'

sys.path.insert(0, str(ROOT / 'src'))
REPORTS.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'CONFIG files: {[f.name for f in CONFIG.glob("*.yaml")]}')

ROOT: /home/thechieft/projects/IAeng
CONFIG files: ['question_types.yaml', 'metrics.yaml', 'business_rules.yaml']


In [10]:
metrics_long  = pd.read_parquet(PROCESSED / 'metrics_long.parquet')
orders_long   = pd.read_parquet(PROCESSED / 'orders_long.parquet')
zone_master   = pd.read_parquet(PROCESSED / 'zone_master.parquet')

# ── load config ────────────────────────────────────────────────────────────
with open(CONFIG / 'metrics.yaml') as f:
    metrics_config = yaml.safe_load(f)

with open(CONFIG / 'business_rules.yaml') as f:
    biz_rules = yaml.safe_load(f)

with open(CONFIG / 'question_types.yaml') as f:
    question_types = yaml.safe_load(f)

print(f'metrics_long: {metrics_long.shape}')
print(f'orders_long : {orders_long.shape}')
print(f'zone_master : {zone_master.shape}')
print(f'Metrics in catalog: {len(metrics_config["metrics"])}')
print(f'Intents in catalog : {len(question_types["question_types"])}')

metrics_long: (104490, 14)
orders_long : (11178, 11)
zone_master : (1244, 13)
Metrics in catalog: 13
Intents in catalog : 8


## 1. Por qué existe la capa semántica

El pipeline (NB 00) produce datos limpios. El EDA (NB 10) produce observaciones.  
**La capa semántica convierte observaciones en reglas interpretables por máquina.**

Sin ella, el insight engine y el bot tendrían que inferir:
- si un valor alto de una métrica es bueno o malo
- qué zonas son comparables entre sí
- qué tipo de pregunta está haciendo el usuario
- cuándo incluir una advertencia de incertidumbre

**Separación de responsabilidades:**

| Capa | Responsabilidad | Artefacto |
|---|---|---|
| Dato | Valores numéricos por zona/semana | parquets en `processed/` |
| Interpretación | Significado de los valores | `config/metrics.yaml` |
| Reglas de negocio | Comparabilidad, thresholds, gobernanza | `config/business_rules.yaml` |
| Intents | Tipos de pregunta soportados | `config/question_types.yaml` |
| Respuestas | Narrativa auditada | NB 40 (chatbot) |

> **Principio guía:** distinguir siempre entre *dato* (observado), *interpretación* (hipótesis de trabajo) y *decisión de negocio* (requiere validación externa).

## 2. Inventario de métricas

**Qué:** tabla maestra que combina el catálogo YAML con estadísticas del dataset real.  
**Por qué:** detectar inconsistencias entre la definición y los datos observados (outliers, escala, cobertura).  
**Resultado esperado:** catálogo enriquecido con flags de violación de escala y gaps de cobertura.

In [11]:
# ── build catalog dataframe from YAML ─────────────────────────────────────
catalog_rows = []
for m in metrics_config['metrics']:
    catalog_rows.append({
        'metric_id':            m['id'],
        'display_name':         m['display_name'],
        'metric_group':         m['metric_group'],
        'source_table':         m['source_table'],
        'scale':                m['scale'],
        'desired_direction':    m['desired_direction'],
        'direction_confidence': m['direction_confidence'],
        'outlier_risk':         m['outlier_risk'],
        'low_coverage_peer':    m.get('low_coverage_peer_groups', False),
    })
catalog_df = pd.DataFrame(catalog_rows)

# ── compute per-metric stats from dataset ─────────────────────────────────
metric_stats = (
    metrics_long
    .dropna(subset=['VALUE'])
    .groupby('METRIC')['VALUE']
    .agg(
        obs_min='min',
        obs_p25=lambda x: x.quantile(0.25),
        obs_median='median',
        obs_p75=lambda x: x.quantile(0.75),
        obs_max='max',
        n_obs='count',
    )
    .reset_index()
    .rename(columns={'METRIC': 'display_name'})
)

catalog_enriched = catalog_df.merge(metric_stats, on='display_name', how='left')

# ── flag scale violations ─────────────────────────────────────────────────
def check_scale_violation(row) -> str:
    if row['scale'] == 'ratio_0_1':
        if pd.notna(row['obs_max']) and row['obs_max'] > 1.05:
            return f'MAX={row["obs_max"]:.3f} excede 1.0'
        if pd.notna(row['obs_min']) and row['obs_min'] < -0.05:
            return f'MIN={row["obs_min"]:.3f} negativo'
    elif row['scale'] == 'free_signed':
        return 'ok_free'
    elif row['scale'] == 'ratio_anomalous':
        return 'documentado_anomalous'
    return 'ok'

catalog_enriched['scale_check'] = catalog_enriched.apply(check_scale_violation, axis=1)

display_cols = ['metric_id', 'display_name', 'metric_group', 'scale',
                'desired_direction', 'direction_confidence',
                'obs_min', 'obs_median', 'obs_max', 'n_obs', 'scale_check']
print(catalog_enriched[display_cols].to_string(index=False))

                                  metric_id                                   display_name              metric_group           scale desired_direction direction_confidence     obs_min  obs_median    obs_max  n_obs           scale_check
                             non_pro_ptc_op                               Non-Pro PTC > OP         conversion_funnel       ratio_0_1  higher_is_better          provisional    0.000000    0.716683   1.000000   8788                    ok
pct_restaurants_sessions_optimal_assortment % Restaurants Sessions With Optimal Assortment        assortment_quality       ratio_0_1  higher_is_better          provisional    0.000000    0.731967   1.000000   8691                    ok
                     pro_adoption_last_week                Pro Adoption (Last Week Status) monetization_subscription       ratio_0_1  higher_is_better          provisional    0.000000    0.283835   1.000000   8636                    ok
                     restaurants_ss_atc_cvr             

## 3. Capacidades analíticas por métrica

No todas las métricas soportan todos los análisis.  
Esta sección materializa las restricciones del catálogo YAML en una matriz de capacidades.

In [12]:
def build_capabilities(m: dict) -> dict:
    direction  = m['desired_direction']
    confidence = m['direction_confidence']
    scale      = m['scale']
    outlier    = m['outlier_risk']
    low_cov    = m.get('low_coverage_peer_groups', False)
    mid        = m['id']

    direction_known = direction in ('higher_is_better', 'lower_is_better')
    stable_scale    = scale == 'ratio_0_1'
    low_outlier     = outlier == 'low'

    return {
        'metric_id':                m['id'],
        'display_name':             m['display_name'],
        'query':                    True,
        'aggregation':              True,
        'trend':                    True,
        'ranking':                  mid != 'lead_penetration' and low_outlier,
        'comparison':               True,
        'anomaly_detection':        direction_known and low_outlier,
        'persistent_deterioration': direction_known,
        'benchmarking':             direction_known and not low_cov and low_outlier,
        'driver_analysis':          direction_known and confidence != 'unknown_pending_validation',
        'caveat':                   (
            'outlier_risk=high — usar MAD/IQR' if outlier == 'high' else
            'cobertura_baja — excluir de peer benchmark' if low_cov else
            'direction_provisional — narrativa neutra' if confidence == 'provisional' else
            ''
        )
    }

caps_rows = [build_capabilities(m) for m in metrics_config['metrics']]
caps_df   = pd.DataFrame(caps_rows)

bool_cols = ['query','aggregation','trend','ranking','comparison',
             'anomaly_detection','persistent_deterioration','benchmarking','driver_analysis']
for c in bool_cols:
    caps_df[c] = caps_df[c].map({True: '✓', False: '✗'})

print(caps_df[['display_name'] + bool_cols + ['caveat']].to_string(index=False))

                                  display_name query aggregation trend ranking comparison anomaly_detection persistent_deterioration benchmarking driver_analysis                                     caveat
                              Non-Pro PTC > OP     ✓           ✓     ✓       ✓          ✓                 ✓                        ✓            ✓               ✓   direction_provisional — narrativa neutra
% Restaurants Sessions With Optimal Assortment     ✓           ✓     ✓       ✓          ✓                 ✓                        ✓            ✓               ✓   direction_provisional — narrativa neutra
               Pro Adoption (Last Week Status)     ✓           ✓     ✓       ✓          ✓                 ✓                        ✓            ✓               ✓   direction_provisional — narrativa neutra
                      Restaurants SS > ATC CVR     ✓           ✓     ✓       ✓          ✓                 ✓                        ✓            ✓               ✓   direction_provis

## 4. Entidades y llaves lógicas

**Qué:** probar empíricamente que `ZONE` sola no es identificador único y documentar la llave compuesta oficial.  
**Por qué:** el bot y los joins deben usar siempre `(COUNTRY, CITY, ZONE)`. Un join por `ZONE` sola mezclaría zonas de distintos mercados.  
**Regla del sistema:** `ZONE_KEY = COUNTRY + '|' + CITY + '|' + ZONE`

In [13]:
zone_keys = (
    metrics_long[['COUNTRY','CITY','ZONE']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# ── is ZONE unique? ────────────────────────────────────────────────────────
zone_name_count = zone_keys.groupby('ZONE').size()
ambiguous_zones = zone_name_count[zone_name_count > 1]

print(f'Unique (COUNTRY, CITY, ZONE) triples  : {len(zone_keys):,}')
print(f'Unique ZONE names                      : {zone_keys["ZONE"].nunique():,}')
print(f'ZONE names appearing in >1 context     : {len(ambiguous_zones):,}')
print()
print('Top 10 ambiguous ZONE names:')
print(ambiguous_zones.sort_values(ascending=False).head(10).to_string())
print()

# ── example: show why ZONE is ambiguous ───────────────────────────────────
example_zone = ambiguous_zones.index[0]
print(f'Example — all occurrences of ZONE="{example_zone}":')
print(zone_keys[zone_keys['ZONE'] == example_zone].to_string(index=False))

# ── build ZONE_KEY in zone_master ─────────────────────────────────────────
zone_master['ZONE_KEY'] = (
    zone_master['COUNTRY'] + '|' +
    zone_master['CITY']    + '|' +
    zone_master['ZONE']
)
assert zone_master['ZONE_KEY'].is_unique, 'ZONE_KEY not unique — investigate'
print(f'\nZONE_KEY is unique across all {len(zone_master):,} zones. OK')

Unique (COUNTRY, CITY, ZONE) triples  : 980
Unique ZONE names                      : 964
ZONE names appearing in >1 context     : 11

Top 10 ambiguous ZONE names:
ZONE
Centro           7
Belgrano         2
Colina           2
Independencia    2
La Molina        2
Miraflores       2
Norte            2
Prado            2
San Miguel       2
San Pedro        2

Example — all occurrences of ZONE="Belgrano":
COUNTRY         CITY     ZONE
     AR      Rosario Belgrano
     AR Buenos Aires Belgrano

ZONE_KEY is unique across all 1,244 zones. OK


## 5. Peer groups y comparabilidad

**Qué:** materializar y evaluar los peer groups definidos en `business_rules.yaml`.  
**Criterio primario:** `(COUNTRY, ZONE_TYPE, ZONE_PRIORITIZATION)` — combina mercado, perfil socioeconómico y prioridad operativa.  
**Umbral mínimo:** 10 zonas por grupo para benchmark confiable.

In [14]:
min_size  = biz_rules['peer_groups']['size_rules']['min_size_for_benchmark']
warn_size = biz_rules['peer_groups']['size_rules']['min_size_warning_threshold']

peer_dims = biz_rules['peer_groups']['primary']['dimensions']  # ['COUNTRY','ZONE_TYPE','ZONE_PRIORITIZATION']

# ── use zones that have metrics (BOTH + ONLY_METRICS) ─────────────────────
zones_with_metrics = zone_master[
    zone_master['COVERAGE_CLASS'].isin(['BOTH', 'ONLY_METRICS'])
].copy()

available_dims = [d for d in peer_dims if d in zones_with_metrics.columns]

peer_groups = (
    zones_with_metrics
    .groupby(available_dims, dropna=False)
    .size()
    .reset_index(name='n_zones')
    .sort_values('n_zones', ascending=False)
)

peer_groups['confidence_level'] = pd.cut(
    peer_groups['n_zones'],
    bins=[-1, warn_size - 1, min_size - 1, float('inf')],
    labels=['too_small', 'low_confidence', 'reliable']
)

print('Peer group distribution by confidence level:')
print(peer_groups['confidence_level'].value_counts().to_string())
print()
print('Largest 10 peer groups:')
print(peer_groups.head(10).to_string(index=False))
print()
print('Smallest / problematic groups (< min_size):')
print(peer_groups[peer_groups['confidence_level'] != 'reliable'].to_string(index=False))

Peer group distribution by confidence level:
confidence_level
reliable          25
too_small         11
low_confidence     7

Largest 10 peer groups:
COUNTRY   ZONE_TYPE ZONE_PRIORITIZATION  n_zones confidence_level
     BR Non Wealthy     Not Prioritized      145         reliable
     MX Non Wealthy         Prioritized      119         reliable
     MX Non Wealthy     Not Prioritized       94         reliable
     MX     Wealthy         Prioritized       57         reliable
     CO Non Wealthy     Not Prioritized       42         reliable
     PE Non Wealthy         Prioritized       39         reliable
     AR Non Wealthy         Prioritized       38         reliable
     CO Non Wealthy         Prioritized       38         reliable
     AR Non Wealthy     Not Prioritized       32         reliable
     EC Non Wealthy     Not Prioritized       29         reliable

Smallest / problematic groups (< min_size):
COUNTRY   ZONE_TYPE ZONE_PRIORITIZATION  n_zones confidence_level
     CL     W

## 6. Taxonomía de intents del sistema

**Qué:** visualizar el catálogo de `question_types.yaml` como tabla de referencia rápida.  
**Uso:** el LLM en NB 40 usará esta taxonomía para clasificar preguntas del usuario antes de despachar a la función semántica correcta.

In [15]:
intent_rows = []
for qt in question_types['question_types']:
    intent_rows.append({
        'intent_id':       qt['id'],
        'display_name':    qt['display_name'],
        'required_params': ', '.join(qt.get('required_params', [])),
        'future_function': qt.get('future_function', 'TBD'),
        'support_level':   qt.get('support_level', 'unknown'),
        'n_caveats':       len(qt.get('caveats', [])),
        'example':         qt.get('examples_from_brief', ['—'])[0],
    })

intents_df = pd.DataFrame(intent_rows)
print(intents_df[['intent_id','display_name','support_level','required_params','example']]
      .to_string(index=False))

           intent_id                  display_name support_level                 required_params                                                                 example
               query              Consulta directa          bien             metric_id, zone_key      ¿Cuál es el Gross Profit UE de Buenos Aires Caballito esta semana?
           aggregate                    Agregación          bien             metric_id, group_by            ¿Cuál es el promedio de Turbo Adoption por país esta semana?
                rank                       Ranking          bien         metric_id, entity_level                              Top 5 zonas por Perfect Orders esta semana
               trend            Tendencia temporal          bien             metric_id, zone_key ¿Cómo ha evolucionado el Pro Adoption en Lima en las últimas 8 semanas?
             compare   Comparación entre segmentos          bien metric_id, segment_a, segment_b    ¿Cómo están las zonas Wealthy vs Non Wealthy en CVR de 

## 7. Reglas de lenguaje del sistema

Estas reglas gobiernan cómo el bot (NB 40) redactará respuestas automáticas.  
El objetivo es narrativa auditada: nunca afirmar más de lo que el dato soporta.

In [16]:
lang = biz_rules['language_rules']

print('=== Reglas de mejora/empeoramiento ===')
print(lang['improvement']['when_to_say'])
print(f'  Safe  : {lang["improvement"]["example_safe"]}')
print(f'  Unsafe: {lang["improvement"]["example_unsafe"]}')

print()
print('=== Términos causales PROHIBIDOS ===')
print(', '.join(lang['causal_language']['forbidden_terms']))

print()
print('=== Términos PERMITIDOS para relaciones entre métricas ===')
print(', '.join(lang['causal_language']['allowed_terms']))

print()
print('=== Cuándo incluir advertencia de incertidumbre ===')
for cond in lang['uncertainty']['always_include_when']:
    print(f'  - {cond}')

print()
print('=== Formato de respuesta con comparación ===')
print(lang['limits']['example'])

=== Reglas de mejora/empeoramiento ===
Solo usar "mejoró" o "empeoró" cuando desired_direction está definido (higher_is_better o lower_is_better) con direction_confidence != unknown_pending_validation. Para métricas con depends o unknown: describir el cambio en términos neutros.

  Safe  : Gross Profit UE aumentó 0.5 puntos WoW en esta zona.
  Unsafe: Esta zona mejoró su rentabilidad. (requiere validación de negocio)

=== Términos causales PROHIBIDOS ===
causó, determinó, explica, genera, produce, es la causa de

=== Términos PERMITIDOS para relaciones entre métricas ===
se asocia con, covaría con, coincide con, es consistente con

=== Cuándo incluir advertencia de incertidumbre ===
  - peer group size < min_size_for_benchmark
  - metric direction_confidence = provisional
  - history_window < min_history_for_trend_analysis

=== Formato de respuesta con comparación ===
"ZONE_X está 3.2% por debajo de la mediana de su peer group (Wealthy / High Priority / MX) en la semana L0W. Baseline: 

## 8. Validación del contrato semántico

10 checks que verifican coherencia entre dataset, catálogo YAML y reglas de negocio.  
Todos deben estar en PASS antes de usar NB 30 o NB 40.

In [17]:
sem_results = {}

def sem_chk(name: str, condition: bool, detail: str = '') -> None:
    status = 'PASS' if condition else 'FAIL'
    sem_results[name] = {'status': status, 'detail': detail}
    icon = '✓' if condition else '✗'
    print(f'  [{icon}] {name}: {status}  {detail}')

dataset_metrics  = set(metrics_long['METRIC'].unique())
catalog_names    = {m['display_name'] for m in metrics_config['metrics']}
catalog_ids      = [m['id'] for m in metrics_config['metrics']]

print('=== Semantic contract checks ===')

sem_chk(
    'all_dataset_metrics_in_catalog',
    dataset_metrics.issubset(catalog_names),
    f'dataset={len(dataset_metrics)}, catalog={len(catalog_names)}, '
    f'missing={dataset_metrics - catalog_names}'
)

sem_chk(
    'no_orphan_metrics_in_catalog',
    catalog_names.issubset(dataset_metrics),
    f'orphans={catalog_names - dataset_metrics}'
)

sem_chk(
    'metric_ids_are_unique',
    len(catalog_ids) == len(set(catalog_ids)),
    f'n_ids={len(catalog_ids)}, n_unique={len(set(catalog_ids))}'
)

sem_chk(
    'all_metrics_have_direction',
    all(m.get('desired_direction') for m in metrics_config['metrics']),
    'all entries have desired_direction field'
)

sem_chk(
    'all_metrics_have_confidence',
    all(m.get('direction_confidence') for m in metrics_config['metrics']),
    'all entries have direction_confidence field'
)

sem_chk(
    'zone_key_is_unique',
    zone_master['ZONE_KEY'].is_unique if 'ZONE_KEY' in zone_master.columns else False,
    f'n_zone_keys={zone_master["ZONE_KEY"].nunique() if "ZONE_KEY" in zone_master.columns else 0}'
)

n_reliable = (peer_groups['confidence_level'] == 'reliable').sum()
sem_chk(
    'peer_groups_have_reliable_groups',
    n_reliable > 0,
    f'reliable_groups={n_reliable}, total_groups={len(peer_groups)}'
)

sem_chk(
    'intents_have_required_params',
    all(qt.get('required_params') for qt in question_types['question_types']),
    f'n_intents={len(question_types["question_types"])}'
)

sem_chk(
    'intents_have_examples',
    all(qt.get('examples_from_brief') for qt in question_types['question_types']),
    ''
)

scale_violations = catalog_enriched[
    ~catalog_enriched['scale_check'].isin(['ok', 'ok_free', 'documentado_anomalous'])
]
sem_chk(
    'scale_violations_documented',
    len(scale_violations) == 0,
    f'violations={list(scale_violations["metric_id"]) if len(scale_violations) > 0 else "none"}'
)

n_pass = sum(1 for v in sem_results.values() if v['status'] == 'PASS')
n_fail = sum(1 for v in sem_results.values() if v['status'] == 'FAIL')
print(f'\nTotal: {n_pass} PASS, {n_fail} FAIL')

=== Semantic contract checks ===
  [✓] all_dataset_metrics_in_catalog: PASS  dataset=13, catalog=13, missing=set()
  [✓] no_orphan_metrics_in_catalog: PASS  orphans=set()
  [✓] metric_ids_are_unique: PASS  n_ids=13, n_unique=13
  [✓] all_metrics_have_direction: PASS  all entries have desired_direction field
  [✓] all_metrics_have_confidence: PASS  all entries have direction_confidence field
  [✓] zone_key_is_unique: PASS  n_zone_keys=1244
  [✓] peer_groups_have_reliable_groups: PASS  reliable_groups=25, total_groups=43
  [✓] intents_have_required_params: PASS  n_intents=8
  [✓] intents_have_examples: PASS  
  [✓] scale_violations_documented: PASS  violations=none

Total: 10 PASS, 0 FAIL


## 9. Exportar reporte semántico

In [18]:
report = {
    'generated_at':    datetime.now().isoformat(),
    'notebook':        '20_reto1_semantic_layer',
    'semantic_checks': sem_results,
    'catalog_summary': {
        'n_metrics_in_catalog':    len(catalog_ids),
        'n_metrics_in_dataset':    len(dataset_metrics),
        'direction_distribution':  (
            catalog_df['desired_direction'].value_counts().to_dict()
        ),
        'confidence_distribution': (
            catalog_df['direction_confidence'].value_counts().to_dict()
        ),
        'outlier_risk_distribution': (
            catalog_df['outlier_risk'].value_counts().to_dict()
        ),
    },
    'peer_group_summary': {
        'total_groups':         len(peer_groups),
        'reliable_groups':      int((peer_groups['confidence_level'] == 'reliable').sum()),
        'low_confidence_groups':int((peer_groups['confidence_level'] == 'low_confidence').sum()),
        'too_small_groups':     int((peer_groups['confidence_level'] == 'too_small').sum()),
        'min_size_threshold':   min_size,
    },
    'intent_summary': {
        'n_intents':         len(question_types['question_types']),
        'support_levels':    {
            qt['id']: qt.get('support_level', 'unknown')
            for qt in question_types['question_types']
        }
    },
    'open_items': [
        'Todas las direcciones de métricas son provisionales — requieren validación de negocio',
        'Thresholds de detectores en business_rules.yaml pendientes de calibración por métrica',
        'lead_penetration excluida de rankings y benchmarks hasta clarificar denominador',
        'turbo_adoption excluida de peer benchmarks por baja cobertura (<70% en algunos grupos)',
        'Peer groups con low_confidence o too_small producirán alertas low_confidence en NB 30',
    ]
}

json_path = REPORTS / 'semantic_layer_report.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(f'JSON → {json_path}')

# ── markdown summary ──────────────────────────────────────────────────────
checks_md = '\n'.join(
    f"| `{k}` | {'✓ PASS' if v['status']=='PASS' else '✗ FAIL'} | {v['detail']} |"
    for k, v in sem_results.items()
)

open_items_md = '\n'.join(f'- {item}' for item in report['open_items'])

md_content = f"""# Semantic Layer Report — Reto 1
Generated: {report['generated_at']}

## Semantic Checks
| Check | Status | Detail |
|---|---|---|
{checks_md}

## Catalog Summary
- Metrics in catalog: {report['catalog_summary']['n_metrics_in_catalog']}
- Metrics in dataset: {report['catalog_summary']['n_metrics_in_dataset']}
- Direction distribution: {report['catalog_summary']['direction_distribution']}
- Direction confidence: {report['catalog_summary']['confidence_distribution']}

## Peer Group Summary
- Total groups: {report['peer_group_summary']['total_groups']}
- Reliable (≥{min_size} zones): {report['peer_group_summary']['reliable_groups']}
- Low confidence: {report['peer_group_summary']['low_confidence_groups']}
- Too small: {report['peer_group_summary']['too_small_groups']}

## Open Items
{open_items_md}
"""

md_path = REPORTS / 'semantic_layer_report.md'
md_path.write_text(md_content, encoding='utf-8')
print(f'MD  → {md_path}')
print('\n=== Semantic layer report exported ===')

JSON → /home/thechieft/projects/IAeng/reports/reto1/semantic_layer_report.json
MD  → /home/thechieft/projects/IAeng/reports/reto1/semantic_layer_report.md

=== Semantic layer report exported ===


## 10. Qué quedó definido y próximos pasos

### Quedó definido en este notebook

| Decisión | Artefacto | Estado |
|---|---|---|
| Catálogo de 13 métricas con escala, dirección y grupos | `config/metrics.yaml` | Provisional |
| Llave lógica de zona: `(COUNTRY, CITY, ZONE)` | Sección 4 + `zone_master['ZONE_KEY']` | Definido |
| Peer group primario: `(COUNTRY, ZONE_TYPE, ZONE_PRIORITIZATION)` | `config/business_rules.yaml` | Provisional |
| 7 intents del sistema con parámetros y funciones futuras | `config/question_types.yaml` | Provisional |
| Reglas de lenguaje: términos permitidos/prohibidos | `config/business_rules.yaml` | Inicial |
| 10 semantic contract checks | Sección 8 | Activo |

### Quedó provisional / pendiente de validación

- **Todas las `desired_direction`** son hipótesis. Ninguna está confirmada con negocio.
- **Thresholds de detectores** (wow_delta=10%, decline_streak=3w, robust_z=2.5) son puntos de partida.
- **Peer groups pequeños** — varios grupos tienen <10 zonas; afectan confiabilidad de benchmarks.
- **lead_penetration** — outlier extremo (393.9); excluida de rankings hasta clarificar denominador.
- **turbo_adoption** — cobertura <70% en algunos grupos; excluida de peer benchmarks.

### Qué habilita esto para NB 30 (Insight Engine)

NB 30 puede ahora:
1. Leer `config/metrics.yaml` para saber qué dirección usar en cada detector.
2. Leer `config/business_rules.yaml` para peer groups, thresholds y reglas temporales.
3. Operar los 4 detectores: `wow_delta`, `decline_streak`, `vs_peer_median`, `robust_z`.
4. Marcar alertas como `low_confidence` cuando peer group < `min_size_for_benchmark`.
5. Excluir `lead_penetration` y `turbo_adoption` de benchmarks peer automáticamente.

> **Antes de usar NB 30 en producción:** validar direcciones de métricas con el área de negocio.